![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 3 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">E3.3 · PCA y t-SNE</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Objetivo** | Programar PCA con la SVD, decidir cuántas componentes conservar, usar PCA dentro de un pipeline y explorar con t-SNE sin sacar conclusiones que el método no permite. |
| **Resultado de aprendizaje** | RDA2 · competencias CG-G2 y CE-G2 (según el sílabo) |
| **Duración** | ≈ 4 h |
| **Teoría** | Manual M3 §6–7 · Animaciones A3.5 y A3.6 · Video V3.2 |
| **Datos** | Sintéticos · **Olivetti faces** (`fetch_olivetti_faces`: 400 rostros de 64 × 64 píxeles) · **digits** (`load_digits`: 1797 dígitos manuscritos de 8 × 8) |

**Niveles:** 1 · PCA por SVD desde cero → 2 · `PCA` y eigenfaces → 3 · digits: PCA frente a t-SNE y PCA en un pipeline supervisado → 4 · reto: un error típico al leer t-SNE.

## 0 · Configuración

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
DIEZ = [UEES["vino"], UEES["azul"], UEES["ocre"], UEES["verde"], UEES["gris"], "#5FB3CE", "#E07A9B", "#6B102C", "#9BBF85", "#1C1A1B"]
print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## Nivel 1 · Desde cero (prelaboratorio)

PCA busca las direcciones de **máxima varianza**. Con la matriz de datos centrada $X_c$ y su descomposición en valores singulares $X_c = U\Sigma V^\top$:
- las columnas de $V$ son las **componentes principales** (direcciones);
- la varianza explicada por la componente $j$ es $\sigma_j^2 / (n-1)$;
- la **proyección** en k componentes es $Z = X_c V_k$ y la **reconstrucción** es $\hat X = Z V_k^\top + \bar x$.

In [ ]:
rng = np.random.default_rng(SEED)
# nube 3D alargada: casi toda la variación vive en un plano inclinado
Z0 = rng.normal(size=(500, 3)) * np.array([3.0, 1.2, 0.25])
rot = np.linalg.qr(rng.normal(size=(3, 3)))[0]
X3 = Z0 @ rot.T + np.array([1.0, -2.0, 0.5])


def pca_svd(X, k):
    """Devuelve (componentes k × p, varianza explicada, proporción explicada, media)."""
    # TODO: centra X, aplica np.linalg.svd(Xc, full_matrices=False) y calcula las varianzas.
    media = ...
    Xc = ...
    U, s, Vt = ...
    var = ...
    return ..., ..., ..., media


def reconstruir(X, comps, media):
    return (X - media) @ comps.T @ comps + media


comps3, var3, prop3, media3 = pca_svd(X3, 3)
print("proporción de varianza explicada:", np.round(prop3, 4))
for k in (1, 2, 3):
    err = np.mean(np.sum((X3 - reconstruir(X3, comps3[:k], media3)) ** 2, axis=1))
    print(f"k = {k}: error medio de reconstrucción {err:.4f}")

**Qué observar.** Dos componentes explican casi toda la varianza: la tercera dirección solo tiene ruido. Reconstruir con k = 2 pierde muy poco, y con k = 3 el error es cero porque no se descarta nada.

## Nivel 2 · Con scikit-learn

In [ ]:
from sklearn.decomposition import PCA

pca3 = PCA(n_components=3).fit(X3)
print("scikit-learn:", np.round(pca3.explained_variance_ratio_, 4))
assert np.allclose(pca3.explained_variance_ratio_, prop3), "Tu varianza explicada no coincide."
assert np.allclose(np.abs(pca3.components_), np.abs(comps3)), "Tus componentes no coinciden (salvo el signo)."
print("✓ Tu PCA coincide con scikit-learn (las componentes pueden diferir en el signo)")

### 2.1 Eigenfaces
Cada rostro de Olivetti es un vector de 4096 píxeles. PCA encuentra "rostros base" (**eigenfaces**) y cada rostro se aproxima como el rostro medio más una combinación de unas pocas de ellas.

In [ ]:
from sklearn.datasets import fetch_olivetti_faces

caras = fetch_olivetti_faces(shuffle=True, random_state=SEED)
Xf = caras.data
pca_f = PCA(n_components=150, random_state=SEED).fit(Xf)
acumulada = np.cumsum(pca_f.explained_variance_ratio_)
print(f"{Xf.shape[0]} rostros × {Xf.shape[1]} píxeles · varianza con 10 / 50 / 150 componentes: "
      f"{acumulada[9]:.1%} / {acumulada[49]:.1%} / {acumulada[149]:.1%}")

fig, axes = plt.subplots(2, 6, figsize=(11, 4.2))
axes[0, 0].imshow(pca_f.mean_.reshape(64, 64), cmap="gray")
axes[0, 0].set_title("rostro medio", fontsize=9)
for j in range(5):
    axes[0, j + 1].imshow(pca_f.components_[j].reshape(64, 64), cmap="RdBu_r")
    axes[0, j + 1].set_title(f"eigenface {j + 1}", fontsize=9)
original = Xf[0]
axes[1, 0].imshow(original.reshape(64, 64), cmap="gray")
axes[1, 0].set_title("original", fontsize=9)
for j, k in enumerate((5, 10, 25, 50, 150)):
    z = (original - pca_f.mean_) @ pca_f.components_[:k].T
    axes[1, j + 1].imshow((z @ pca_f.components_[:k] + pca_f.mean_).reshape(64, 64), cmap="gray")
    axes[1, j + 1].set_title(f"k = {k}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

**Qué observar.** Con 50 de 4096 dimensiones ya se reconoce el rostro: la información está muy concentrada. Las primeras eigenfaces capturan iluminación y forma general; las siguientes, detalles.

## Nivel 3 · digits: PCA frente a t-SNE

Cada dígito manuscrito es un vector de 64 píxeles (8 × 8). Comparamos una proyección lineal (PCA a 2D) con t-SNE, que preserva **vecindarios locales**.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

dig = load_digits()
Xd, yd = dig.data, dig.target
pca_d = PCA(random_state=SEED).fit(Xd)
acum_d = np.cumsum(pca_d.explained_variance_ratio_)
k90 = int(np.searchsorted(acum_d, 0.90) + 1)
print(f"digits: {Xd.shape} · 2 componentes explican {acum_d[1]:.1%} · se necesitan {k90} para el 90 %")

emb = {}
tiempos = {}
for perp in (5, 30, 50):
    t0 = time.perf_counter()
    emb[perp] = TSNE(n_components=2, perplexity=perp, init="pca", random_state=SEED).fit_transform(Xd)
    tiempos[perp] = time.perf_counter() - t0
print("segundos por t-SNE:", {p: round(t, 1) for p, t in tiempos.items()})

fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
Z2 = pca_d.transform(Xd)[:, :2]
vistas = [("PCA (2 componentes)", Z2)] + [(f"t-SNE · perplejidad {p}", emb[p]) for p in (5, 30, 50)]
for ax, (titulo, Z) in zip(axes, vistas):
    ax.scatter(Z[:, 0], Z[:, 1], c=[DIEZ[c] for c in yd], s=4)
    for d in range(10):
        cx, cy = np.median(Z[yd == d], axis=0)
        ax.text(cx, cy, str(d), fontsize=11, weight="bold", ha="center", va="center", color="#1C1A1B")
    ax.set(title=titulo, xticks=[], yticks=[])
plt.tight_layout()
plt.show()

**Qué observar.** PCA en 2D explica menos del 30 % de la varianza y mezcla varios dígitos; t-SNE separa casi todos los dígitos en islas. Con perplejidad 5 aparecen sub-grupos artificiales; 30 y 50 dan mapas parecidos entre sí. t-SNE es excelente para **explorar**, pero no produce variables para un modelo: no tiene `transform` para datos nuevos y su resultado cambia con los parámetros.

### 3.1 PCA como paso de un pipeline supervisado

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
filas = []
for k in (5, 10, 20, 30, None):
    pasos = [StandardScaler()] + ([PCA(n_components=k, random_state=SEED)] if k else []) + [LogisticRegression(max_iter=3000)]
    t0 = time.perf_counter()
    acc = cross_val_score(make_pipeline(*pasos), Xd, yd, cv=cv)
    filas.append({"componentes": k or "todas (64)", "exactitud CV": f"{acc.mean():.3f} ± {acc.std():.3f}", "segundos": round(time.perf_counter() - t0, 2)})
tabla_pipe = pd.DataFrame(filas)
tabla_pipe

**Qué observar.** Con 30 componentes la exactitud baja menos de dos puntos (0.955 frente a 0.971) usando menos de la mitad de las variables; con 5 componentes cae a ≈ 0.82. Aquí PCA no acelera nada, porque el modelo ya es rápido: su valor aparece con miles de variables muy correlacionadas (como los píxeles de los rostros). Y va **dentro** del pipeline: se ajusta solo con los folds de entrenamiento, igual que el escalador (semana 1).

## Nivel 4 · Reto: lo que t-SNE no permite leer

Creamos tres grupos con **tamaños y distancias muy distintas**: uno compacto, uno disperso (10 veces más ancho) y uno muy lejano. ¿Los respeta el mapa de t-SNE?

In [ ]:
r = np.random.default_rng(SEED)
g1 = r.normal([0, 0, 0, 0, 0], 0.3, size=(150, 5))
g2 = r.normal([4, 0, 0, 0, 0], 3.0, size=(150, 5))
g3 = r.normal([60, 0, 0, 0, 0], 0.3, size=(150, 5))
Xt = np.vstack([g1, g2, g3])
gt = np.repeat([0, 1, 2], 150)
Zt = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(Xt)
Zp = PCA(n_components=2).fit_transform(Xt)


def dispersion(Z, g):
    return [float(np.mean(np.linalg.norm(Z[g == k] - Z[g == k].mean(axis=0), axis=1))) for k in range(3)]


def distancia_centros(Z, g):
    c = [Z[g == k].mean(axis=0) for k in range(3)]
    return np.linalg.norm(c[0] - c[1]), np.linalg.norm(c[0] - c[2])


filas = []
for nombre, Z in (("datos originales (5D)", Xt), ("PCA 2D", Zp), ("t-SNE 2D", Zt)):
    disp = dispersion(Z, gt)
    d01, d02 = distancia_centros(Z, gt)
    filas.append({"espacio": nombre, "dispersión disperso / compacto": round(disp[1] / disp[0], 1),
                  "distancia al lejano / al vecino": round(d02 / d01, 1)})
tabla_tsne = pd.DataFrame(filas).set_index("espacio")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, (titulo, Z) in zip(axes, (("PCA 2D (respeta tamaños y distancias)", Zp), ("t-SNE 2D (no los respeta)", Zt))):
    for k, (c, n) in enumerate(zip((UEES["vino"], UEES["azul"], UEES["ocre"]), ("compacto", "disperso", "lejano"))):
        ax.scatter(Z[gt == k, 0], Z[gt == k, 1], s=6, c=c, label=n)
    ax.set(title=titulo, xticks=[], yticks=[])
    ax.legend()
plt.tight_layout()
plt.show()
tabla_tsne

**Qué observar.** En los datos, el grupo disperso es ≈ 10 veces más ancho que el compacto y el lejano está ≈ 17 veces más lejos que el vecino. PCA conserva esas proporciones; t-SNE las **aplana**: dibuja grupos de tamaño parecido y distancias entre grupos que no significan nada. En un mapa de t-SNE solo se lee **quién está cerca de quién**, no cuánto miden los grupos ni cuán lejos están.

## Autoverificación

In [ ]:
assert np.allclose(pca3.explained_variance_ratio_, prop3)
assert prop3[:2].sum() > 0.95, "Dos componentes deberían explicar casi toda la varianza de la nube 3D."
assert acumulada[49] > 0.8, "50 eigenfaces deberían explicar más del 80 % de la varianza."
assert tabla_tsne.loc["PCA 2D", "distancia al lejano / al vecino"] > 3 * tabla_tsne.loc["t-SNE 2D", "distancia al lejano / al vecino"], \
    "t-SNE debería aplanar la diferencia de distancias entre grupos."
print("✓ E3.3 completo")

## Lista de cotejo (autoevaluación)

- [ ] Tu PCA por SVD coincide con scikit-learn.
- [ ] Elegiste el número de componentes con la varianza acumulada y la reconstrucción.
- [ ] Comparaste PCA con t-SNE en digits con varias perplejidades.
- [ ] Usaste PCA dentro de un pipeline supervisado.
- [ ] Demostraste qué no se puede leer en un mapa de t-SNE.

**Reflexión:** ¿qué afirmación sobre un mapa de t-SNE has visto (o hecho) que no estaba justificada?

**Cómo te prepara para la Tarea 3:** la tarea pide PCA con varianza explicada y reconstrucción, y t-SNE solo exploratorio con al menos dos perplejidades y advertencias.

## Desafío opcional con IA agéntica · ¿Qué conserva cada mapa?

**Objetivo.** Comparar PCA, t-SNE e Isomap midiendo cuánto conserva cada uno los vecindarios.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
En el notebook resuelto E3.3 (PCA y t-SNE con digits), agrega una sección que compare en 2D PCA, t-SNE con perplejidades 5, 30 y 50, e Isomap. Para cada mapa calcula sklearn.manifold.trustworthiness con 10 vecinos y muéstralos lado a lado. Explica en español qué conserva y qué no conserva cada mapa, y qué conclusiones no se pueden sacar de ellos.
```

**Cómo verificar el resultado**

- Todos los mapas usan las mismas muestras y la misma semilla.
- Se muestran al menos dos perplejidades de t-SNE.
- La explicación no interpreta tamaños ni distancias entre grupos de t-SNE.

**Declara el uso de IA** (norma f del sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.